# 02a - Train DL Methods

Run after `01_preprocessing.ipynb` and before `02_run_methods.ipynb`.

1. trains one DL method across folds on Modal, all folds concurrently,
2. writes checkpoints to the `segmentation-data` Volume.

Classical methods (1a, 2, 3c, 4) skip this notebook entirely; they run locally in
`02_run_methods.ipynb`.

Training happens in Modal containers, not in this kernel. This notebook only launches the
runs and collects their histories. Requires the Volume to be seeded with
`data/processed/` first (see `00_setup.ipynb`).

In [ ]:
import os, pathlib, sys

root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.paths import resolve_roots
DATA_ROOT, OUTPUT_ROOT = resolve_roots()

print('Repo root  :', root)
print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
import modal

from src.modal_app import DEFAULT_GPU, app, train_fold

# DL model classes are currently NotImplementedError stubs, so this notebook cannot run end to end yet.
METHOD = 'unet'
FOLDS = [0]  # Stage 0 uses one fold; full sweep sets list(range(5))
SEED = 42
GPU = DEFAULT_GPU  # override per run, e.g. 'A100'; see modal_app.benchmark_gpus

# Paths are resolved inside the container against the mounted Volume, so
# PROCESSED_DIRS is no longer built here.
print(f'{METHOD}: folds {FOLDS}, seed {SEED}, gpu {GPU}')

In [ ]:
histories = {}

# starmap fans the folds out to concurrent containers, so wall-clock is one
# training run rather than len(FOLDS) of them. A fresh model is built inside
# each container, so folds never share one.
#
# Uncomment once the method's model class is implemented.
# with modal.enable_output(), app.run():
#     runs = train_fold.with_options(gpu=GPU).starmap(
#         [(METHOD, fold, SEED) for fold in FOLDS]
#     )
#     for run in runs:
#         histories[run['fold']] = run['history']
#         best = min(step['val_loss'] for step in run['history'])
#         print(f"fold {run['fold']}: best val_loss={best:.4f} -> {run['checkpoint']}")

In [ ]:
import matplotlib.pyplot as plt

# Plot train and val loss per epoch across folds
# for fold, history in histories.items():
#     epochs = [step['epoch'] for step in history]
#     train_loss = [step['train_loss'] for step in history]
#     val_loss = [step['val_loss'] for step in history]
#     plt.plot(epochs, train_loss, label=f'Fold {fold} Train')
#     plt.plot(epochs, val_loss, label=f'Fold {fold} Val')
# plt.xlabel('Epoch')
# plt.ylabel('Loss')
# plt.legend()
# plt.show()